# Análisis de Precio y Abastecimiento — Papa Superior Corabastos (2019–2025)

**Objetivo:** Identificar qué variables explican el precio de la papa Superior en Corabastos, partiendo de la hipótesis de que el abastecimiento es el principal determinante.

**Fuentes de datos:**
- Abastecimiento: SIPSA — DANE (2013–2025)
- Precios: SIPSA — DANE (2013–2026)
- IPC Alimentos: Banco de la República (2019–2025)

**Estructura del análisis:**
1. Limpieza y preparación de datos
2. Análisis descriptivo
3. Correlación precio vs abastecimiento
4. Incorporación del IPC y precio real
5. Estacionalidad
6. Modelos explicativos del precio

---
## 0. Librerías y configuración

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from statsmodels.tsa.seasonal import seasonal_decompose
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'sans-serif'

# --- RUTAS ---
RAW       = r"C:\Users\anaso\OneDrive\Desktop\Analisis de Datos 2026\proyecto_papa_superior_corabastos\data\raw"
PROCESSED = r"C:\Users\anaso\OneDrive\Desktop\Analisis de Datos 2026\proyecto_papa_superior_corabastos\data\processed"

meses_orden = ['Enero','Febrero','Marzo','Abril','Mayo','Junio',
               'Julio','Agosto','Septiembre','Octubre','Noviembre','Diciembre']
mes_num = {m: str(i+1).zfill(2) for i, m in enumerate(meses_orden)}

print('✅ Librerías cargadas correctamente')

✅ Librerías cargadas correctamente


---
## 1. Limpieza y preparación de datos

In [ ]:
# Cargar datos crudos
df_abast_raw  = pd.read_csv(f"{RAW}\\ABASTECIMIENTO_PAPA_2013_2025.csv", encoding='latin1')
df_precios_raw = pd.read_excel(f"{RAW}\\Base_Precios_historica_13_2026.xlsx", sheet_name='Base Papa')

print('Abastecimiento:', df_abast_raw.shape)
print('Precios:', df_precios_raw.shape)

In [ ]:
# Limpiar abastecimiento
df_abast = df_abast_raw.copy()
df_abast.columns = ['Central','fecha','Cod_Depto','Cod_Mun','Departamento',
                    'Municipio','variedad','semana','Año','CantKg','mes','Toneladas']
df_abast['mes']   = df_abast['mes'].str.capitalize()
df_abast['fecha'] = pd.to_datetime(df_abast['fecha'], format='%d/%m/%Y')

df_abast_filtrado = df_abast[
    (df_abast['Central']  == 'Corabastos') &
    (df_abast['variedad'] == 'Superior') &
    (df_abast['Año'].between(2019, 2025))
].copy()

print('✅ Abastecimiento filtrado:', df_abast_filtrado.shape)

In [ ]:
# Limpiar precios
df_precios = df_precios_raw.copy()
df_precios['mes'] = df_precios['mes'].str.capitalize()

df_precios_filtrado = df_precios[
    (df_precios['ciudad']  == 'Bogotá D.C.') &
    (df_precios['variedad'] == 'Superior') &
    (df_precios['year'].between(2019, 2025))
].copy()

df_precios_filtrado = df_precios_filtrado[
    ['year','mes','semana','fecha3','precio']
].rename(columns={'year':'Año','fecha3':'fecha'})

print('✅ Precios filtrados:', df_precios_filtrado.shape)

In [ ]:
# Agregar a nivel mensual y crear dataset maestro
abast_mensual   = df_abast_filtrado.groupby(['Año','mes'], as_index=False)['Toneladas'].sum()
precios_mensual = df_precios_filtrado.groupby(['Año','mes'], as_index=False)['precio'].mean()\
                                     .rename(columns={'precio':'precio_promedio'})

df_maestro = pd.merge(abast_mensual, precios_mensual, on=['Año','mes'], how='inner')
df_maestro['mes'] = pd.Categorical(df_maestro['mes'], categories=meses_orden, ordered=True)
df_maestro = df_maestro.sort_values(['Año','mes']).reset_index(drop=True)

# Guardar
df_maestro.to_csv(f"{PROCESSED}\\papa_superior_corabastos_2019_2025.csv", index=False)
df_maestro.to_parquet(f"{PROCESSED}\\papa_superior_corabastos_2019_2025.parquet", index=False)

print('✅ Dataset maestro creado:', df_maestro.shape)
df_maestro.head(12)

---
## 2. Análisis Descriptivo

In [ ]:
# Estadísticas básicas por año
resumen = df_maestro.groupby('Año').agg(
    precio_min      = ('precio_promedio','min'),
    precio_max      = ('precio_promedio','max'),
    precio_medio    = ('precio_promedio','mean'),
    precio_std      = ('precio_promedio','std'),
    toneladas_total = ('Toneladas','sum'),
    toneladas_media = ('Toneladas','mean')
).round(2)

resumen.to_csv(f"{PROCESSED}\\resumen_anual.csv")
print(resumen.to_string())

In [ ]:
# Evolución anual del precio
precio_anual = df_maestro.groupby('Año')['precio_promedio'].mean().reset_index()

fig, ax = plt.subplots(figsize=(10,5))
ax.plot(precio_anual['Año'], precio_anual['precio_promedio'],
        marker='o', linewidth=2.5, color='#E07B39', markersize=8)
for _, row in precio_anual.iterrows():
    ax.annotate(f"${row['precio_promedio']:,.0f}",
                xy=(row['Año'], row['precio_promedio']),
                xytext=(0,12), textcoords='offset points', ha='center', fontsize=9)
ax.set_title('Precio promedio anual — Papa Superior Corabastos (2019–2025)', fontsize=13, fontweight='bold')
ax.set_xlabel('Año')
ax.set_ylabel('Precio promedio ($/kg)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))
ax.set_xticks(precio_anual['Año'])
plt.tight_layout()
plt.savefig(f"{PROCESSED}\\grafico_precio_anual.png", bbox_inches='tight')
plt.show()

In [ ]:
# Abastecimiento anual
abast_anual = df_maestro.groupby('Año')['Toneladas'].sum().reset_index()

fig, ax = plt.subplots(figsize=(10,5))
bars = ax.bar(abast_anual['Año'], abast_anual['Toneladas'], color='#4C9BE8', width=0.6, edgecolor='white')
for bar in bars:
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1500,
            f"{bar.get_height():,.0f} t", ha='center', fontsize=9)
ax.set_title('Abastecimiento total anual — Papa Superior Corabastos (2019–2025)', fontsize=13, fontweight='bold')
ax.set_xlabel('Año')
ax.set_ylabel('Toneladas')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:,.0f}'))
ax.set_xticks(abast_anual['Año'])
plt.tight_layout()
plt.savefig(f"{PROCESSED}\\grafico_abastecimiento_anual.png", bbox_inches='tight')
plt.show()

In [ ]:
# Precio vs Abastecimiento doble eje
fig, ax1 = plt.subplots(figsize=(12,5))
ax1.plot(precio_anual['Año'], precio_anual['precio_promedio'],
         marker='o', color='#E07B39', linewidth=2.5, label='Precio promedio')
ax1.set_ylabel('Precio promedio ($/kg)', color='#E07B39')
ax1.tick_params(axis='y', labelcolor='#E07B39')
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))
ax2 = ax1.twinx()
ax2.bar(abast_anual['Año'], abast_anual['Toneladas'], color='#4C9BE8', alpha=0.4, width=0.6, label='Toneladas')
ax2.set_ylabel('Toneladas', color='#4C9BE8')
ax2.tick_params(axis='y', labelcolor='#4C9BE8')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:,.0f}'))
ax1.set_title('Precio vs Abastecimiento anual — Papa Superior Corabastos (2019–2025)', fontsize=13, fontweight='bold')
ax1.set_xlabel('Año')
ax1.set_xticks(precio_anual['Año'])
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1+lines2, labels1+labels2, loc='upper left')
plt.tight_layout()
plt.savefig(f"{PROCESSED}\\grafico_precio_vs_abastecimiento.png", bbox_inches='tight')
plt.show()

In [ ]:
# Heatmap precio por año y mes
pivot_precio = df_maestro.pivot_table(index='Año', columns='mes', values='precio_promedio')
fig, ax = plt.subplots(figsize=(14,5))
sns.heatmap(pivot_precio, annot=True, fmt='.0f', cmap='YlOrRd',
            linewidths=0.5, ax=ax, cbar_kws={'label':'$/kg'})
ax.set_title('Heatmap de precio promedio mensual — Papa Superior Corabastos (2019–2025)', fontsize=13, fontweight='bold')
ax.set_xlabel('Mes')
ax.set_ylabel('Año')
plt.tight_layout()
plt.savefig(f"{PROCESSED}\\grafico_heatmap_precio.png", bbox_inches='tight')
plt.show()

---
## 3. Correlación Precio vs Abastecimiento

**Hipótesis inicial:** A mayor abastecimiento, menor precio.

In [ ]:
# Correlaciones mensuales
pearson_m,  p_pearson_m  = stats.pearsonr(df_maestro['Toneladas'], df_maestro['precio_promedio'])
spearman_m, p_spearman_m = stats.spearmanr(df_maestro['Toneladas'], df_maestro['precio_promedio'])

# Rezago 1 mes
df_maestro['precio_lag1'] = df_maestro['precio_promedio'].shift(1)
df_lag = df_maestro.dropna(subset=['precio_lag1'])
pearson_lag, p_lag = stats.pearsonr(df_lag['Toneladas'], df_lag['precio_lag1'])

resumen_corr = pd.DataFrame({
    'Nivel'        : ['Mensual','Rezago 1 mes'],
    'Pearson r'    : [round(pearson_m,4), round(pearson_lag,4)],
    'Pearson p'    : [round(p_pearson_m,4), round(p_lag,4)],
    'Spearman r'   : [round(spearman_m,4), '-'],
    'Spearman p'   : [round(p_spearman_m,4), '-'],
    'Significativo': ['Sí (Spearman)','No']
})

resumen_corr.to_csv(f"{PROCESSED}\\resumen_correlaciones.csv", index=False)
print(resumen_corr.to_string(index=False))
print()
print('Conclusión: La correlación es débil → el abastecimiento solo no explica el precio.')

In [ ]:
# Scatter mensual precio vs abastecimiento
años    = sorted(df_maestro['Año'].unique())
colores = sns.color_palette('tab10', len(años))

fig, ax = plt.subplots(figsize=(9,6))
for año, color in zip(años, colores):
    sub = df_maestro[df_maestro['Año']==año]
    ax.scatter(sub['Toneladas'], sub['precio_promedio'], label=str(año), color=color, s=60, alpha=0.85)
m, b, *_ = stats.linregress(df_maestro['Toneladas'], df_maestro['precio_promedio'])
x_range  = pd.Series([df_maestro['Toneladas'].min(), df_maestro['Toneladas'].max()])
ax.plot(x_range, m*x_range+b, color='black', linewidth=1.5, linestyle='--', label=f'Tendencia (r={pearson_m:.2f})')
ax.set_title('Correlación mensual: Precio vs Abastecimiento\nPapa Superior — Corabastos (2019–2025)', fontsize=13, fontweight='bold')
ax.set_xlabel('Toneladas abastecidas (mensual)')
ax.set_ylabel('Precio promedio ($/kg)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:,.0f}'))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))
ax.legend(title='Año', bbox_to_anchor=(1.01,1), loc='upper left')
plt.tight_layout()
plt.savefig(f"{PROCESSED}\\grafico_scatter_mensual.png", bbox_inches='tight')
plt.show()

---
## 4. IPC Alimentos y Precio Real

**Nueva pregunta:** Si el abastecimiento no explica bien el precio, ¿podría ser la inflación?

In [ ]:
# Cargar y construir índice IPC alimentos
df_ipc = pd.read_excel(f"{RAW}\\graficador_seriesf.xlsx", sheet_name='Datos', skiprows=2)
df_ipc.columns = ['fecha','ipc_alimentos']
df_ipc = df_ipc[df_ipc['fecha'].notna() & (df_ipc['fecha'] != '')]
df_ipc = df_ipc[~df_ipc['fecha'].astype(str).str.contains('Descargado')]
df_ipc['fecha'] = pd.to_datetime(df_ipc['fecha'], format='%d/%m/%Y')
df_ipc['ipc_alimentos'] = df_ipc['ipc_alimentos'].astype(str).str.replace(',','.').astype(float)
df_ipc = df_ipc[df_ipc['fecha'].dt.year.between(2019,2025)].reset_index(drop=True)

# Construir índice acumulado base 100 = enero 2019
df_ipc['factor_mensual'] = (1 + df_ipc['ipc_alimentos'] / 100) ** (1/12)
df_ipc['indice'] = 100.0
for i in range(1, len(df_ipc)):
    df_ipc.loc[i,'indice'] = df_ipc.loc[i-1,'indice'] * df_ipc.loc[i,'factor_mensual']

print(f'IPC construido — índice diciembre 2025: {df_ipc.iloc[-1]["indice"]:.2f}')
print('Interpretación: los alimentos costaban 88.65% más en dic-2025 que en ene-2019')

In [ ]:
# Unir IPC con dataset maestro
df = df_maestro.copy()
df['fecha'] = pd.to_datetime(
    df['Año'].astype(str) + '-' + df['mes'].astype(str).map(mes_num) + '-01'
)
df_ipc['fecha_merge'] = df_ipc['fecha'].values.astype('datetime64[M]').astype('datetime64[D]')
df['fecha_merge']     = df['fecha'].values.astype('datetime64[M]').astype('datetime64[D]')
df = pd.merge(df, df_ipc[['fecha_merge','ipc_alimentos','indice']], on='fecha_merge', how='left')

# Precio real deflactado
df['precio_real'] = (df['precio_promedio'] / df['indice']) * 100

# Guardar dataset enriquecido
df.to_csv(f"{PROCESSED}\\papa_superior_corabastos_con_ipc.csv", index=False)
df.to_parquet(f"{PROCESSED}\\papa_superior_corabastos_con_ipc.parquet", index=False)

print('✅ Dataset con IPC guardado:', df.shape)
df[['Año','mes','precio_promedio','indice','precio_real','ipc_alimentos']].head(12)

In [ ]:
# Precio nominal vs precio real
df_anual = df.groupby('Año').agg(
    precio_nominal = ('precio_promedio','mean'),
    precio_real    = ('precio_real','mean')
).reset_index()

fig, ax = plt.subplots(figsize=(11,6))
ax.plot(df_anual['Año'], df_anual['precio_nominal'], marker='o', linewidth=2.5,
        color='#E07B39', label='Precio nominal ($/kg)')
ax.plot(df_anual['Año'], df_anual['precio_real'], marker='s', linewidth=2.5,
        color='#4C9BE8', linestyle='--', label='Precio real (base ene-2019)')
for _, row in df_anual.iterrows():
    ax.annotate(f"${row['precio_nominal']:,.0f}", xy=(row['Año'], row['precio_nominal']),
                xytext=(0,12), textcoords='offset points', ha='center', fontsize=8, color='#E07B39')
    ax.annotate(f"${row['precio_real']:,.0f}", xy=(row['Año'], row['precio_real']),
                xytext=(0,-18), textcoords='offset points', ha='center', fontsize=8, color='#4C9BE8')
ax.set_title('Precio nominal vs precio real — Papa Superior Corabastos (2019–2025)', fontsize=13, fontweight='bold')
ax.set_xlabel('Año')
ax.set_ylabel('$/kg')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))
ax.set_xticks(df_anual['Año'])
ax.legend()
plt.tight_layout()
plt.savefig(f"{PROCESSED}\\grafico_precio_nominal_vs_real.png", bbox_inches='tight')
plt.show()

---
## 5. Estacionalidad

In [ ]:
# Preparar serie de tiempo
df_ts = df.copy().set_index('fecha')
df_ts['mes_str'] = df_ts['mes'].astype(str)

# Descomposición estacional
descomp = seasonal_decompose(df_ts['precio_promedio'], model='additive', period=12)

fig, axes = plt.subplots(4, 1, figsize=(13,10), sharex=True)
descomp.observed.plot(ax=axes[0], color='#333333', linewidth=1.5)
axes[0].set_ylabel('Precio original')
axes[0].set_title('Descomposición de la serie de precios — Papa Superior Corabastos (2019–2025)', fontsize=13, fontweight='bold')
descomp.trend.plot(ax=axes[1], color='#E07B39', linewidth=2)
axes[1].set_ylabel('Tendencia')
descomp.seasonal.plot(ax=axes[2], color='#4C9BE8', linewidth=1.5)
axes[2].set_ylabel('Estacionalidad')
descomp.resid.plot(ax=axes[3], color='#888888', linewidth=1, marker='o', markersize=3)
axes[3].set_ylabel('Residuo')
axes[3].set_xlabel('Fecha')
for ax in axes:
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))
plt.tight_layout()
plt.savefig(f"{PROCESSED}\\grafico_descomposicion_precio.png", bbox_inches='tight')
plt.show()

In [ ]:
# Precio promedio por mes
estacional_precio = df_ts.groupby('mes_str')['precio_promedio'].mean().reindex(meses_orden)
estacional_abast  = df_ts.groupby('mes_str')['Toneladas'].mean().reindex(meses_orden)

fig, ax = plt.subplots(figsize=(12,5))
colores_p = ['#E07B39' if v==estacional_precio.max() else '#4C9BE8' if v==estacional_precio.min()
             else '#A8C8E8' for v in estacional_precio.values]
bars = ax.bar(meses_orden, estacional_precio.values, color=colores_p, edgecolor='white', width=0.7)
for bar, val in zip(bars, estacional_precio.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+15,
            f'${val:,.0f}', ha='center', fontsize=9)
ax.set_title('Precio promedio histórico por mes — Papa Superior Corabastos (2019–2025)', fontsize=13, fontweight='bold')
ax.set_xlabel('Mes')
ax.set_ylabel('Precio promedio ($/kg)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig(f"{PROCESSED}\\grafico_estacional_precio.png", bbox_inches='tight')
plt.show()

print(f"Mes más caro:   {estacional_precio.idxmax()} (${estacional_precio.max():,.0f})")
print(f"Mes más barato: {estacional_precio.idxmin()} (${estacional_precio.min():,.0f})")

In [ ]:
# Evolución mensual por año superpuesta
colores = sns.color_palette('tab10', len(años))
fig, ax = plt.subplots(figsize=(13,6))
for año, color in zip(años, colores):
    sub  = df_ts[df_ts['Año']==año]
    vals = sub['precio_promedio'].reindex(
        pd.date_range(f'{año}-01-01', f'{año}-12-01', freq='MS')
    ).values
    ax.plot(meses_orden[:len(vals)], vals, marker='o', linewidth=2,
            label=str(año), color=color, markersize=5)
ax.set_title('Evolución mensual del precio por año — Papa Superior Corabastos', fontsize=13, fontweight='bold')
ax.set_xlabel('Mes')
ax.set_ylabel('Precio promedio ($/kg)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))
ax.legend(title='Año', bbox_to_anchor=(1.01,1), loc='upper left')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig(f"{PROCESSED}\\grafico_lineas_por_año.png", bbox_inches='tight')
plt.show()

# Guardar índice estacional
estacional_df = pd.DataFrame({
    'mes'                         : meses_orden,
    'precio_promedio_historico'   : estacional_precio.values.round(2),
    'toneladas_promedio_historico': estacional_abast.values.round(2)
})
estacional_df.to_csv(f"{PROCESSED}\\indice_estacional.csv", index=False)
print('✅ Índice estacional guardado')

---
## 6. Modelos Explicativos del Precio

**Pregunta:** ¿Podemos predecir el precio usando abastecimiento, IPC y el precio anterior?

In [ ]:
# Preparar variables
df['mes_num']     = df['mes'].cat.codes + 1
df['precio_lag1'] = df['precio_promedio'].shift(1)
df['precio_lag2'] = df['precio_promedio'].shift(2)
df['precio_ma3']  = df['precio_promedio'].rolling(3).mean()

df_model = df.dropna().copy()
y = df_model['precio_promedio']

# Tres modelos progresivos
X1 = df_model[['Toneladas']]
X2 = df_model[['Toneladas','ipc_alimentos']]
X3 = df_model[['Toneladas','ipc_alimentos','precio_lag1','mes_num']]

m1 = LinearRegression().fit(X1, y)
m2 = LinearRegression().fit(X2, y)
m3 = LinearRegression().fit(X3, y)

def metricas(modelo, X, y):
    pred = modelo.predict(X)
    return {
        'R²'  : round(modelo.score(X,y), 4),
        'MAE' : round(mean_absolute_error(y,pred), 0),
        'RMSE': round(np.sqrt(mean_squared_error(y,pred)), 0)
    }

resumen_modelos = pd.DataFrame({
    'Modelo': ['M1: solo abastecimiento',
               'M2: abastecimiento + IPC',
               'M3: completo (+ lag + estacionalidad)'],
    **{k: [metricas(m,X,y)[k] for m,X in [(m1,X1),(m2,X2),(m3,X3)]]
       for k in ['R²','MAE','RMSE']}
})

resumen_modelos.to_csv(f"{PROCESSED}\\resumen_modelos.csv", index=False)
print(resumen_modelos.to_string(index=False))

In [ ]:
# Coeficientes modelo 3
coef_df = pd.DataFrame({
    'Variable'     : ['Toneladas','ipc_alimentos','precio_lag1','mes_num'],
    'Coeficiente'  : m3.coef_.round(4),
    'Interpretación': [
        'Por cada 1.000 ton más → precio baja $10 aprox.',
        'Por cada 1% más de inflación alimentos → precio sube $6.6',
        'El precio del mes anterior explica 84% del precio actual',
        'Cada mes avanzado → precio baja $5 (efecto estacional)'
    ]
})
print(f"Intercepto: {m3.intercept_:.2f}")
print()
print(coef_df.to_string(index=False))

In [ ]:
# Comparación de los 3 modelos
pred_m1 = m1.predict(X1)
pred_m2 = m2.predict(X2)
pred_m3 = m3.predict(X3)

fig, ax = plt.subplots(figsize=(13,5))
ax.plot(df_model['fecha'], y.values, color='#333333', linewidth=2.5, label='Precio real')
ax.plot(df_model['fecha'], pred_m1, color='#A8C8E8', linewidth=1.5, linestyle=':', label=f'M1 — solo abastecimiento (R²={round(m1.score(X1,y),2)})')
ax.plot(df_model['fecha'], pred_m2, color='#4C9BE8', linewidth=1.5, linestyle='--', label=f'M2 — + IPC (R²={round(m2.score(X2,y),2)})')
ax.plot(df_model['fecha'], pred_m3, color='#E07B39', linewidth=2, linestyle='--', label=f'M3 — completo (R²={round(m3.score(X3,y),2)})')
ax.set_title('Comparación de modelos — Papa Superior Corabastos (2019–2025)', fontsize=13, fontweight='bold')
ax.set_xlabel('Fecha')
ax.set_ylabel('$/kg')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))
ax.legend(loc='upper left')
plt.tight_layout()
plt.savefig(f"{PROCESSED}\\grafico_comparacion_modelos.png", bbox_inches='tight')
plt.show()

In [ ]:
# Promedios móviles
df['ma3']  = df['precio_promedio'].rolling(3,  center=False).mean()
df['ma6']  = df['precio_promedio'].rolling(6,  center=False).mean()
df['ma12'] = df['precio_promedio'].rolling(12, center=False).mean()

fig, ax = plt.subplots(figsize=(13,5))
ax.plot(df['fecha'], df['precio_promedio'], color='#BBBBBB', linewidth=1.5, label='Precio mensual', alpha=0.8)
ax.plot(df['fecha'], df['ma3'],  color='#4C9BE8', linewidth=2,   label='Promedio móvil 3 meses')
ax.plot(df['fecha'], df['ma6'],  color='#E07B39', linewidth=2,   label='Promedio móvil 6 meses')
ax.plot(df['fecha'], df['ma12'], color='#2E7D32', linewidth=2.5, label='Promedio móvil 12 meses (tendencia)')
ax.set_title('Promedios móviles del precio — Papa Superior Corabastos (2019–2025)', fontsize=13, fontweight='bold')
ax.set_xlabel('Fecha')
ax.set_ylabel('$/kg')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))
ax.legend()
plt.tight_layout()
plt.savefig(f"{PROCESSED}\\grafico_promedios_moviles.png", bbox_inches='tight')
plt.show()

# Guardar predicciones
df_model['precio_predicho_m3'] = pred_m3
df_model[['Año','mes','fecha','precio_promedio','precio_predicho_m3']].to_csv(
    f"{PROCESSED}\\predicciones_modelo3.csv", index=False
)
print('✅ Todos los archivos guardados correctamente')